In [1]:
# If needed:
# !pip install numpy pandas scikit-learn


import warnings
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")

In [2]:
books=pd.read_csv(r"D:\Data Science_bat\Recommendation_system\data\Books.csv")
rating=pd.read_csv(r"D:\Data Science_bat\Recommendation_system\data\Ratings.csv")
users=pd.read_csv(r"D:\Data Science_bat\Recommendation_system\data\Users.csv")


In [3]:
books.head()

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...


In [29]:
rating.head()

,User-ID,ISBN,Book-Rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0
3,276729,052165615X,3
4,276729,0521795028,6


In [30]:
users

,User-ID,Location,Age
0,1,"nyc, new york, usa",NaN
1,2,"stockton, california, usa",18.0
2,3,"moscow, yukon territory, russia",NaN
3,4,"porto, v.n.gaia, portugal",17.0
4,5,"farnborough, hants, united kingdom",NaN
...,...,...,...
278853,278854,"portland, oregon, usa",NaN
278854,278855,"tacoma, washington, united kingdom",50.0
278855,278856,"brampton, ontario, canada",NaN
278856,278857,"knoxville, tennessee, usa",NaN


In [4]:
def clean_data(books, ratings, users):
    books.columns = [c.strip() for c in books.columns]
    books = books.rename(columns={
        "Book-Title": "title",
        "Book-Author": "author",
        "Year-Of-Publication": "year",
        "Publisher": "publisher",
    })

    books["year"] = pd.to_numeric(books["year"], errors="coerce")
    books = books.dropna(subset=["ISBN", "title"]).drop_duplicates("ISBN")

    ratings = ratings.rename(columns={
        "User-ID": "user_id",
        "Book-Rating": "rating",
    })
    ratings = ratings.dropna()
    ratings = ratings[ratings["rating"] > 0]

    users = users.rename(columns={"User-ID": "user_id"})
    users = users.dropna(subset=["user_id"])

    return books, ratings, users

In [5]:
books, ratings, users = clean_data(books, rating, users)

In [7]:
merged = ratings.merge(books[["ISBN","title"]], on="ISBN")

In [9]:
merged.groupby("user_id")["rating"].count() >= 10

user_id
8         False
9         False
12        False
14        False
16        False
          ...  
278846    False
278849    False
278851     True
278852    False
278854    False
Name: rating, Length: 68091, dtype: bool

In [ ]:
popular_books = merged.groupby("title")["rating"].count() >= 50

popular_books_df = merged[merged["title"].isin(popular_books[popular_books].index)]
# popular_books_df['title'].value_counts()

title
The Lovely Bones: A Novel                                                         707
Wild Animus                                                                       581
The Da Vinci Code                                                                 494
The Secret Life of Bees                                                           406
The Nanny Diaries: A Novel                                                        393
                                                                                 ... 
Shattered                                                                          50
The Wailing Wind                                                                   50
A Beautiful Mind: The Life of Mathematical Genius and Nobel Laureate John Nash     50
Acceptable Risk                                                                    50
Atonement: A Novel                                                                 50
Name: count, Length: 651, dtype: int64

In [18]:
def build_pivot(books, ratings, min_user_ratings=10, min_book_ratings=50):
    merged = ratings.merge(books[["ISBN","title"]], on="ISBN")

    active_users = (
        merged.groupby("user_id")["rating"].count() >= min_user_ratings
    )
    merged = merged[merged["user_id"].isin(active_users[active_users].index)]

    popular_books = (
        merged.groupby("title")["rating"].count() >= min_book_ratings
    )
    merged = merged[merged["title"].isin(popular_books[popular_books].index)]

    pivot = merged.pivot_table(index="title", columns="user_id", values="rating")
    pivot.fillna(0, inplace=True)

    print("Pivot shape:", pivot.shape)
    return pivot

In [21]:
user_item_pivot = build_pivot(books, ratings)

Pivot shape: (315, 5026)


In [20]:
def build_similarity(pivot):
    similarity = cosine_similarity(pivot)
    print("Similarity shape:", similarity.shape)
    return similarity

In [25]:
similarity = build_similarity(user_item_pivot)

Similarity shape: (315, 315)


In [24]:
user_item_pivot.index

Index(['1984', '1st to Die: A Novel', '2nd Chance', 'A Bend in the Road',
       'A Child Called \It\": One Child's Courage to Survive"',
       'A Heartbreaking Work of Staggering Genius',
       'A Is for Alibi (Kinsey Millhone Mysteries (Paperback))',
       'A Map of the World', 'A Painted House', 'A Prayer for Owen Meany',
       ...
       'While I Was Gone', 'Whispers', 'White Oleander : A Novel',
       'White Oleander : A Novel (Oprah's Book Club)',
       'Wicked: The Life and Times of the Wicked Witch of the West',
       'Wild Animus', 'Wizard and Glass (The Dark Tower, Book 4)',
       'Wuthering Heights', 'Year of Wonders', '\O\" Is for Outlaw"'],
      dtype='object', name='title', length=315)

In [28]:
similarity[0][1]

np.float64(0.038472160853498874)

In [30]:
sorted(
        enumerate(similarity[0]),
        key=lambda x: x[1],
        reverse=True
    )[:6+1]

[(0, np.float64(0.9999999999999994)),
 (24, np.float64(0.23237614147423302)),
 (44, np.float64(0.19836873228818658)),
 (207, np.float64(0.13262497803343096)),
 (285, np.float64(0.12701043059270542)),
 (233, np.float64(0.12511410338162843)),
 (232, np.float64(0.12286777742031796))]

In [31]:
def similar_books(book_title, pivot, similarity, books, top_n=6):
    if book_title not in pivot.index:
        print("Book not found.")
        return []

    idx = list(pivot.index).index(book_title)
    scores = sorted(
        enumerate(similarity[idx]),
        key=lambda x: x[1],
        reverse=True
    )[1:top_n+1]

    recommendations = []
    for i, score in scores:
        title = pivot.index[i]
        meta = books[books["title"] == title]

        recommendations.append({
            "title": title,
            "author": meta["author"].values[0],
            "score": round(score, 3)
        })

    return recommendations

In [33]:
similar_books(book_title='While I Was Gone', pivot=user_item_pivot, similarity=similarity, books=books)

[{'title': 'Snow Falling on Cedars',
  'author': 'David Guterson',
  'score': np.float64(0.168)},
 {'title': 'The Bluest Eye',
  'author': 'Toni Morrison',
  'score': np.float64(0.166)},
 {'title': 'River, Cross My Heart',
  'author': 'Breena Clarke',
  'score': np.float64(0.164)},
 {'title': 'We Were the Mulvaneys',
  'author': 'Joyce Carol Oates',
  'score': np.float64(0.159)},
 {'title': 'What Looks Like Crazy On An Ordinary Day',
  'author': 'Pearl Cleage',
  'score': np.float64(0.14)},
 {'title': 'Jewel', 'author': 'Bret Lott', 'score': np.float64(0.136)}]

In [35]:
user_item_pivot_transposed = user_item_pivot.T

title,1984,1st to Die: A Novel,2nd Chance,A Bend in the Road,"A Child Called \It\"": One Child's Courage to Survive""",A Heartbreaking Work of Staggering Genius,A Is for Alibi (Kinsey Millhone Mysteries (Paperback)),A Map of the World,A Painted House,A Prayer for Owen Meany,...,While I Was Gone,Whispers,White Oleander : A Novel,White Oleander : A Novel (Oprah's Book Club),Wicked: The Life and Times of the Wicked Witch of the West,Wild Animus,"Wizard and Glass (The Dark Tower, Book 4)",Wuthering Heights,Year of Wonders,"\O\"" Is for Outlaw"""
user_id,,,,,,,,,,,,,,,,,,,,,
242,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
243,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,7.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
254,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
388,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
446,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
278535,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.0
278554,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
278582,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [41]:
scaler = StandardScaler()
X = scaler.fit_transform(user_item_pivot_transposed)

kmeans = KMeans(
        n_clusters=3,
        random_state=42,
        n_init=15
    )

labels = kmeans.fit_predict(X)
user_cluster = dict(zip(user_item_pivot_transposed.index, labels))




In [42]:
pd.Series(user_cluster).value_counts()

0    4552
2     421
1      53
Name: count, dtype: int64

In [7]:
def cluster_users(pivot, n_clusters=5):
    user_matrix = pivot.T

    scaler = StandardScaler()
    X = scaler.fit_transform(user_matrix)

    kmeans = KMeans(
        n_clusters=n_clusters,
        random_state=42,
        n_init=15
    )

    labels = kmeans.fit_predict(X)
    user_cluster = dict(zip(user_matrix.index, labels))

    return labels, user_cluster, user_matrix

In [43]:
user_cluster.keys()

dict_keys([242, 243, 254, 388, 446, 503, 505, 507, 638, 709, 805, 882, 900, 929, 1025, 1075, 1129, 1131, 1155, 1211, 1424, 1435, 1548, 1585, 1674, 1733, 1848, 1903, 2010, 2012, 2030, 2033, 2041, 2103, 2110, 2179, 2276, 2288, 2313, 2337, 2358, 2766, 2793, 2799, 2891, 2977, 3145, 3167, 3329, 3331, 3346, 3363, 3371, 3373, 3418, 3538, 3556, 3578, 3640, 3675, 3719, 3757, 3827, 3840, 3917, 3923, 4017, 4092, 4131, 4157, 4197, 4221, 4334, 4385, 4600, 4622, 4785, 4795, 4938, 5037, 5385, 5439, 5476, 5490, 5499, 5539, 5582, 5609, 5728, 5899, 5903, 5956, 6073, 6115, 6238, 6242, 6251, 6280, 6323, 6431, 6532, 6537, 6543, 6563, 6566, 6575, 6577, 6611, 6772, 7082, 7105, 7125, 7134, 7158, 7210, 7283, 7286, 7312, 7346, 7352, 7371, 7426, 7620, 7841, 7864, 7887, 7913, 7915, 7958, 7980, 8019, 8066, 8067, 8234, 8245, 8253, 8255, 8362, 8454, 8529, 8680, 8681, 8687, 8734, 8782, 8801, 8864, 8870, 8872, 8935, 8936, 8937, 9177, 9227, 9354, 9381, 9417, 9574, 9613, 9660, 9747, 9856, 9908, 10030, 10118, 10199, 1031

In [45]:
user_cluster.items()

dict_items([(242, np.int32(0)), (243, np.int32(0)), (254, np.int32(0)), (388, np.int32(0)), (446, np.int32(0)), (503, np.int32(0)), (505, np.int32(0)), (507, np.int32(0)), (638, np.int32(2)), (709, np.int32(0)), (805, np.int32(0)), (882, np.int32(0)), (900, np.int32(0)), (929, np.int32(0)), (1025, np.int32(0)), (1075, np.int32(0)), (1129, np.int32(0)), (1131, np.int32(0)), (1155, np.int32(0)), (1211, np.int32(0)), (1424, np.int32(0)), (1435, np.int32(0)), (1548, np.int32(0)), (1585, np.int32(0)), (1674, np.int32(0)), (1733, np.int32(0)), (1848, np.int32(0)), (1903, np.int32(0)), (2010, np.int32(0)), (2012, np.int32(0)), (2030, np.int32(0)), (2033, np.int32(0)), (2041, np.int32(0)), (2103, np.int32(0)), (2110, np.int32(0)), (2179, np.int32(0)), (2276, np.int32(0)), (2288, np.int32(0)), (2313, np.int32(0)), (2337, np.int32(0)), (2358, np.int32(0)), (2766, np.int32(0)), (2793, np.int32(0)), (2799, np.int32(0)), (2891, np.int32(0)), (2977, np.int32(0)), (3145, np.int32(0)), (3167, np.int32

In [46]:
# user_cluster[267830]
cluster_id = user_cluster[267830]
[user for user, cluster in user_cluster.items() if cluster == cluster_id and user != 267830]

[242,
 243,
 254,
 388,
 446,
 503,
 505,
 507,
 709,
 805,
 882,
 900,
 929,
 1025,
 1075,
 1129,
 1131,
 1155,
 1211,
 1424,
 1435,
 1548,
 1585,
 1674,
 1733,
 1848,
 1903,
 2010,
 2012,
 2030,
 2033,
 2041,
 2103,
 2110,
 2179,
 2276,
 2288,
 2313,
 2337,
 2358,
 2766,
 2793,
 2799,
 2891,
 2977,
 3145,
 3167,
 3329,
 3331,
 3346,
 3363,
 3371,
 3373,
 3418,
 3538,
 3556,
 3578,
 3640,
 3675,
 3719,
 3757,
 3827,
 3840,
 3917,
 3923,
 4092,
 4131,
 4157,
 4197,
 4221,
 4334,
 4385,
 4600,
 4622,
 4785,
 4795,
 4938,
 5037,
 5385,
 5439,
 5476,
 5490,
 5499,
 5539,
 5609,
 5728,
 5899,
 5903,
 5956,
 6073,
 6115,
 6238,
 6280,
 6323,
 6431,
 6532,
 6537,
 6566,
 6577,
 6611,
 6772,
 7082,
 7105,
 7134,
 7210,
 7286,
 7312,
 7352,
 7371,
 7426,
 7620,
 7841,
 7864,
 7887,
 7913,
 7958,
 7980,
 8019,
 8066,
 8234,
 8255,
 8362,
 8529,
 8680,
 8681,
 8687,
 8734,
 8782,
 8801,
 8864,
 8870,
 8872,
 8935,
 8936,
 8937,
 9177,
 9227,
 9354,
 9381,
 9417,
 9574,
 9613,
 9660,
 9747,
 9856

In [48]:
user_item_pivot_transposed.loc[267830].values.reshape(1,-1)

array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 8., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 8., 9., 6., 8., 0., 8., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 9., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 

In [53]:
# set(user_item_pivot_transposed.loc[267830][user_item_pivot_transposed.loc[267830] > 0].index)
user_item_pivot_transposed.loc[267830][user_item_pivot_transposed.loc[267830] > 0].index

Index(['Beloved', 'Harry Potter and the Chamber of Secrets (Book 2)',
       'Harry Potter and the Goblet of Fire (Book 4)',
       'Harry Potter and the Order of the Phoenix (Book 5)',
       'Harry Potter and the Prisoner of Azkaban (Book 3)',
       'Harry Potter and the Sorcerer's Stone (Harry Potter (Paperback))',
       'Me Talk Pretty One Day'],
      dtype='object', name='title')

In [54]:
def recommend_for_user(user_id, user_matrix, user_cluster, books, ratings, top_n=8):

    if user_id not in user_matrix.index:
        print("User not in dataset.")
        return []

    cluster_id = user_cluster[user_id]
    peers = [u for u, c in user_cluster.items()
             if c == cluster_id and u != user_id]

    target_vec = user_matrix.loc[user_id].values.reshape(1,-1)
    peer_matrix = user_matrix.loc[peers]

    sims = cosine_similarity(target_vec, peer_matrix)[0]

    read_books = set(user_matrix.loc[user_id][user_matrix.loc[user_id] > 0].index)
    unread = [b for b in user_matrix.columns if b not in read_books]

    results = []

    for title in unread:
        peer_ratings = peer_matrix[title].values
        mask = peer_ratings > 0

        if mask.sum() == 0:
            continue

        weighted_score = np.dot(sims[mask], peer_ratings[mask]) / (
            np.abs(sims[mask]).sum() + 1e-9
        )

        results.append((title, weighted_score))

    results.sort(key=lambda x: x[1], reverse=True)

    return results[:top_n]

In [64]:
user_item_pivot_transposed

title,1984,1st to Die: A Novel,2nd Chance,A Bend in the Road,"A Child Called \It\"": One Child's Courage to Survive""",A Heartbreaking Work of Staggering Genius,A Is for Alibi (Kinsey Millhone Mysteries (Paperback)),A Map of the World,A Painted House,A Prayer for Owen Meany,...,While I Was Gone,Whispers,White Oleander : A Novel,White Oleander : A Novel (Oprah's Book Club),Wicked: The Life and Times of the Wicked Witch of the West,Wild Animus,"Wizard and Glass (The Dark Tower, Book 4)",Wuthering Heights,Year of Wonders,"\O\"" Is for Outlaw"""
user_id,,,,,,,,,,,,,,,,,,,,,
242,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
243,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,7.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
254,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
388,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
446,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
278535,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.0
278554,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
278582,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [72]:
recommended_books = recommend_for_user(243,user_item_pivot_transposed, user_cluster, books,rating)
recommended_books


[('Anne of Green Gables (Anne of Green Gables Novels (Paperback))',
  np.float64(9.999999925754828)),
 ('The Queen of the Damned (Vampire Chronicles (Paperback))',
  np.float64(9.999999881511153)),
 ("Harry Potter and the Sorcerer's Stone (Book 1)",
  np.float64(9.454269108787448)),
 ('The Two Towers (The Lord of the Rings, Part 2)',
  np.float64(9.395893101217178)),
 ('The Thorn Birds', np.float64(9.362322608400454)),
 ('White Oleander : A Novel', np.float64(9.323988157834346)),
 ('Wuthering Heights', np.float64(9.314654841267554)),
 ('From the Corner of His Eye', np.float64(9.307099012964892))]

In [88]:
for book in recommended_books:
    print(book[0])

    books[books['title'] == "The Thorn Birds"]['Image-URL-S']

Anne of Green Gables (Anne of Green Gables Novels (Paperback))
The Queen of the Damned (Vampire Chronicles (Paperback))
Harry Potter and the Sorcerer's Stone (Book 1)
The Two Towers (The Lord of the Rings, Part 2)
The Thorn Birds
White Oleander : A Novel
Wuthering Heights
From the Corner of His Eye


In [85]:
from IPython.display import HTML, display


image_path = "https://images.amazon.com/images/P/0380018179.0"
display(HTML(f'<img src="{image_path}" style="max-width:700px; border:1px solid #ccc;">'))


In [84]:
books[books['title'] == "The Thorn Birds"]['Image-URL-S']

1209     http://images.amazon.com/images/P/0380018179.0...
14662    http://images.amazon.com/images/P/0380563908.0...
79790    http://images.amazon.com/images/P/0060129565.0...
Name: Image-URL-S, dtype: object

In [ ]:

from IPython.display import Image, display

display(Image(filename="path/to/image.png"))


In [13]:
books, rating, users = clean_data(books, rating, users)

pivot = build_pivot(books, rating)
similarity = build_similarity(pivot)

labels, user_cluster, user_matrix = cluster_users(pivot)

Pivot shape: (315, 5026)
Similarity shape: (315, 315)


In [23]:
similar_books("The Notebook", pivot, similarity, books)

[{'title': 'Message in a Bottle',
  'author': 'Nicholas Sparks',
  'score': np.float64(0.209)},
 {'title': 'A Walk to Remember',
  'author': 'Nicholas Sparks',
  'score': np.float64(0.204)},
 {'title': "Suzanne's Diary for Nicholas",
  'author': 'James Patterson',
  'score': np.float64(0.186)},
 {'title': 'The Rescue',
  'author': 'Nicholas Sparks',
  'score': np.float64(0.184)},
 {'title': 'The Wedding',
  'author': 'Nicholas Sparks',
  'score': np.float64(0.176)},
 {'title': 'The Five People You Meet in Heaven',
  'author': 'Mitch Albom',
  'score': np.float64(0.139)}]

In [56]:
recommend_for_user(242, user_matrix, user_cluster, books, rating)

NameError: name 'user_matrix' is not defined